# Notebook 02 — RAPID Array CREP Analysis

Compute all four CREP components (C, R, E, P) from the RAPID-calibrated
AMOC time series and verify that $\Gamma \approx 0.251$.

| Component | Physical signal |
|-----------|------------------|
| C | AR(1) autocorrelation — critical slowing down |
| R | Freshwater transport Fov at 34°S (van Westen 2024) |
| E | Short/long variance ratio — instability emergence |
| P | Permutation entropy — signal orderliness |

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from amoc_utac.crep_amoc import CREPAmocTensor
from amoc_utac.fingerprint import AmocFingerprintIndex
from amoc_utac.freshwater import FreshwaterTransport
from amoc_utac.rapid_loader import RapidLoader

loader = RapidLoader(seed=42)
years, amoc = loader.synthetic_annual(start=1950, end=2023)

fp  = AmocFingerprintIndex(seed=42)
fwt = FreshwaterTransport(rng=np.random.default_rng(42))
crep = CREPAmocTensor()

ar1 = fp.autocorrelation_ar1(amoc, window=30)
pe  = fp.permutation_entropy(amoc, window=30)
fov = fwt.timeseries(amoc, noise_scale=0.0)

C = crep.compute_c(ar1)
R = crep.compute_r(float(fov[-1]))
E = crep.compute_e(amoc)
P = crep.compute_p(pe)
G = crep.compute_gamma(C, R, E, P)

print(f'C (CSD):         {C:.4f}')
print(f'R (Fov signal):  {R:.4f}')
print(f'E (variance):    {E:.4f}')
print(f'P (perm. ent.):  {P:.4f}')
print(f'Γ_AMOC:          {G:.4f}  (target: 0.251)')

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

axes[0].plot(years, amoc, color='steelblue', lw=1.5)
axes[0].set_ylabel('AMOC [Sv]')
axes[0].set_title('RAPID-calibrated AMOC strength')

axes[1].plot(years, ar1, color='darkorange', lw=1.5, label='AR(1) — CSD indicator')
axes[1].axhline(0.8, color='red', ls=':', lw=1, label='Warning threshold')
axes[1].set_ylabel('AR(1) coefficient')
axes[1].legend(loc='upper left')

axes[2].plot(years, fov, color='teal', lw=1.5, label='Fov [Sv]')
axes[2].axhline(0, color='black', ls='--', lw=1)
axes[2].set_ylabel('Fov [Sv]')
axes[2].set_xlabel('Year')
axes[2].legend()

plt.tight_layout()
plt.savefig('rapid_crep_analysis.png', dpi=150)
plt.show()